# __Demo: Building a Personal Prompt Library with Hugging Face__

## __Steps to Perform:__
Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`

Step 2: Define a Function to Get Completion

Step 3: Define the Prompt Library Structure

Step 4: Populate the Library With 5 Reusable Engineering Prompts

Step 5: Validate Every Entry by Running It

Step 6: Save the Library to a Versioned File

### __Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`__
- Installs required libraries (`transformers`, `torch`, `accelerate`).
- Initializes `Qwen/Qwen2.5-0.5B-Instruct` using half-precision (`torch.float16`) on GPU without requiring API keys or license approvals.

In [1]:
!pip install -q transformers torch accelerate

import os
import torch
from transformers import pipeline

# Initialize pipeline on GPU with float16
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.


### __Step 2: Define a Function to Get Completion__
- Sends the prompt formatted as a chat message to the Hugging Face pipeline.
- Uses greedy decoding (`do_sample=False`) for deterministic, reproducible software engineering completions.

In [2]:
def get_completion(prompt, max_tokens=250):
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=max_tokens, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

print("✅ Completion helper ready.")

✅ Completion helper ready.


### __Step 3: Define the Prompt Library Structure__
A prompt that works well is worth keeping. Each library entry records the task it solves, the exact prompt text, and (once run) a verified example of its output -- so the pattern can be reused with confidence rather than reconstructed from memory every time.

In [3]:
import json
import datetime

prompt_library = []

def add_to_library(task, category, prompt_text):
    prompt_library.append({
        "task": task,
        "category": category,
        "prompt": prompt_text.strip(),
        "example_output": None,
        "validated": False,
    })

print("✅ Prompt library structure and helper initialized.")

✅ Prompt library structure and helper initialized.


### __Step 4: Populate the Library With 5 Reusable Engineering Prompts__
These reuse the five prompt patterns from the previous demo (code generation, test design, PR review, documentation, deployment scripts), organized here as a library rather than one-off prompts.

In [4]:
add_to_library(
    task="Scaffold a log-severity parser",
    category="code_generation",
    prompt_text="""
    Write a Python function with this signature: def parse_severity(log_line: str) -> str.
    Constraints: it must return one of "CRITICAL", "ERROR", "WARN", or "INFO" based on the first
    matching keyword found in the log line (case-insensitive); return "UNKNOWN" if none match.
    Example: parse_severity("2026-09-10 ERROR failed to connect") should return "ERROR".
    Return only the function code.
    """,
)

add_to_library(
    task="Design tests for the severity parser",
    category="test_design",
    prompt_text="""
    Write pytest unit tests for a function parse_severity(log_line) that returns the first matching
    severity keyword (CRITICAL, ERROR, WARN, INFO) or UNKNOWN. Cover the normal case, the no-match
    case, and a case-insensitivity check, stating the assertion expected for each.
    """,
)

add_to_library(
    task="Review a small diff for correctness, security, and style",
    category="pr_review",
    prompt_text="""
    Review this diff. Report correctness issues, security issues, and style issues as three separate
    lists, citing the line number for each.

    1  def parse_severity(log_line):
    2      for level in ["CRITICAL", "ERROR", "WARN", "INFO"]:
    3          if level in log_line:
    4              return level
    5      return "UNKNOWN"
    """,
)

add_to_library(
    task="Document the severity parser",
    category="documentation",
    prompt_text="""
    Write a Google-style docstring for this function, covering Args, Returns, and any exceptions it
    raises. Do not invent behavior that is not in the code.

    def parse_severity(log_line: str) -> str:
        for level in ["CRITICAL", "ERROR", "WARN", "INFO"]:
            if level in log_line.upper():
                return level
        return "UNKNOWN"
    """,
)

add_to_library(
    task="Write a staging deployment script",
    category="deployment_scripts",
    prompt_text="""
    Write a bash script that deploys a Docker container to a staging environment (not production).
    Constraints: it must pull the image tagged "staging-latest", stop and remove any existing
    container named "alerts-api-staging" first, and print a confirmation message naming the
    environment it deployed to.
    """,
)

print(f"Library currently has {len(prompt_library)} entries.")

Library currently has 5 entries.


### __Step 5: Validate Every Entry by Running It__
A prompt only belongs in the library once it has actually been run and its output checked -- an unvalidated prompt is just a guess. Run every entry and record its output.

In [5]:
for entry in prompt_library:
    output = get_completion(entry["prompt"], max_tokens=250)
    entry["example_output"] = output
    entry["validated"] = True
    print(f"=== {entry['task']} ({entry['category']}) ===")
    print(output)
    print()

=== Scaffold a log-severity parser (code_generation) ===
```python
def parse_severity(log_line: str) -> str:
    """
    Parses a log line and returns the severity level as a string.

    Args:
        log_line (str): The log line to be parsed.

    Returns:
        str: The severity level as a string ("CRITICAL", "ERROR", "WARN", "INFO", or "UNKNOWN").
    """
    # Split the log line into components based on spaces
    parts = log_line.split()
    
    # Check for the first component that matches any of the keywords
    for part in parts:
        if part.lower() in ["critical", "error", "warn", "info"]:
            return part
    
    # If no matching keyword is found, return "UNKNOWN"
    return "UNKNOWN"
```

This solution defines a function `parse_severity` that takes a single argument `log_line`, which is expected to be a string representing a log entry. The function splits the log line into its components based on spaces and then checks each component against predefined keyword

### __Step 6: Save the Library to a Versioned File__
Version it like code: save the library with a timestamp so improvements to a prompt can be compared against the version they replaced, rather than silently overwritten.

In [6]:
library_export = {
    "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "entries": prompt_library,
}

with open("prompt_library.json", "w") as f:
    json.dump(library_export, f, indent=2)

print("Saved", len(prompt_library), "validated prompts to prompt_library.json")

Saved 5 validated prompts to prompt_library.json


## __Conclusion__
This turns five one-off prompts into a reusable, versioned library: every entry records the task it solves, its exact wording, and a validated example output, organized by engineering task so the right pattern is easy to find under pressure. The next lesson's applied practice reuses this same structure to build the capstone project's own PROMPTS.md.